In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from pathlib import Path

# 以下のハイパーパラメータを、Optunaで探索しました。
HIDDEN1 = 256
HIDDEN2 = 64
CELU_ALPHA = 2.5
INITIAL_LR = 0.001
WEIGHT_DECAY = 0.025
HUBER_DELTA = 10
TRAIN_BATCH_SIZE = 64
MAX_EPOCHS = 5000

DATA_DIR = Path("dataset")
for data_file in sorted(DATA_DIR.glob("*.csv")):
    print(data_file)

In [ ]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Load data
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

# Drop unneeded columns and handle missing values
train = train.drop(columns=["Unnamed: 12", "id"])  # Dropping unnecessary columns

base_columns = [
    col for col in train.columns
    if col in test.columns
    and col not in {"id", "DIC", "Unnamed: 12"}
]

X = train[base_columns].copy()
X_test = test[base_columns].copy()
y = train["DIC"]


# Find common columns between train and test (excluding "DIC" from train)
common_columns = X.columns.tolist()

# Split the training data 
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalize the data only with train dataset
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
test_scaled = scaler.transform(X_test)

# 訓練データの平均・標準偏差で、目的変数 DIC を標準化する
target_mean = y_train.mean()
target_std = y_train.std(ddof=0)
y_train_scaled = (y_train - target_mean) / target_std
y_val_scaled = (y_val - target_mean) / target_std

In [ ]:
from torch.utils.data import Dataset, DataLoader

class OceanChemistryDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = OceanChemistryDataset(X_train, y_train_scaled)
val_dataset = OceanChemistryDataset(X_val, y_val_scaled)

train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


In [ ]:
class MLPModel(nn.Module):
    def __init__(self, input_size):
        super(MLPModel, self).__init__()
        self.fc1 = nn.Linear(input_size, HIDDEN1)
        self.activation1 = nn.CELU(alpha=CELU_ALPHA)
        self.fc2 = nn.Linear(HIDDEN1, HIDDEN2)
        self.activation2 = nn.CELU(alpha=CELU_ALPHA)
        self.fc3 = nn.Linear(HIDDEN2, 1)  # Output layer for regression

    def forward(self, x):
        x = self.fc1(x)
        x = self.activation1(x)
        x = self.fc2(x)
        x = self.activation2(x)
        x = self.fc3(x)
        return x

# Initialize the model
model = MLPModel(input_size=X_train.shape[1])
print(X_train.shape)


In [ ]:
import torch.optim as optim
from sklearn.metrics import r2_score

# Loss function and optimizer
criterion = nn.HuberLoss(delta=HUBER_DELTA / target_std)
mse_metric = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=INITIAL_LR, weight_decay=WEIGHT_DECAY)

# 学習率のスケジューラ
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)


# 予測誤差MSE・決定係数R^2 の計算
# 目的関数にHuber Lossを用いたが、精度確認のためMSEも知りたいので、別途計算する。
def evaluate_loader(model, loader):
    predictions = []
    targets = []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            predictions.append(model(X_batch).squeeze(-1))
            targets.append(y_batch)
    predictions = torch.cat(predictions)
    targets = torch.cat(targets)
    mse = mse_metric(predictions, targets).item() * target_std**2  # 元の DIC 単位での MSE
    r2 = r2_score(targets.cpu().numpy(), predictions.cpu().numpy())
    return mse, r2


# Training function
def train_model(model, train_loader, val_loader, epochs=MAX_EPOCHS):
    history = {
        "train_mse": [], "val_mse": [],
        "train_r2": [], "val_r2": [],
        "learning_rate": []
    }

    for epoch in range(epochs):
        model.train()

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch).squeeze(-1)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        train_mse, train_r2 = evaluate_loader(model, train_loader)
        val_mse, val_r2 = evaluate_loader(model, val_loader)

        lr = optimizer.param_groups[0]["lr"]

        history["train_mse"].append(train_mse)
        history["val_mse"].append(val_mse)
        history["train_r2"].append(train_r2)
        history["val_r2"].append(val_r2)
        history["learning_rate"].append(lr)

        # 学習率の更新
        scheduler.step()

        if epoch % 100 == 0:
            print(f"Epoch {epoch+1}/{epochs}, "
                  f"Train MSE: {train_mse:.4f}, "
                  f"Validation MSE: {val_mse:.4f}, "
                  f"Validation R²: {val_r2:.4f}, "
                  f"LR: {lr:.8g}")

    return history


# Train the model
history = train_model(model, train_loader, val_loader, epochs=MAX_EPOCHS)

# Validation MSEが最低のエポック
best_epoch = np.argmin(history["val_mse"]) + 1

print(f"Final Validation MSE: {history['val_mse'][-1]:.6f}")
print(f"Final Validation R²: {history['val_r2'][-1]:.6f}")
print(f"Best Validation MSE: {min(history['val_mse']):.6f} (Epoch {best_epoch})")


In [ ]:
# Convert the test set into a torch tensor
test_tensor = torch.tensor(test_scaled, dtype=torch.float32)

# Set the model to evaluation mode
model.eval()

# Make predictions
with torch.no_grad():
    predictions = model(test_tensor).squeeze().numpy() * target_std + target_mean

# Prepare submission
submission = pd.DataFrame({"id": test["id"].to_numpy(), "DIC": predictions})
submission.to_csv("submission.csv", index=False)


In [ ]:
# 学習曲線の出力

import matplotlib.pyplot as plt

# 表示するエポックの範囲
start_epoch = 1000
end_epoch = min(3000, len(history["train_mse"]))
if start_epoch > end_epoch:
    raise ValueError("開始エポックが学習履歴の範囲外です。")

epoch_slice = slice(start_epoch - 1, end_epoch)
epochs = range(start_epoch, end_epoch + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MSEとR²を描画
for ax, metric, title in zip(axes, ["mse", "r2"], ["MSE", "R²"]):
    ax.plot(epochs, history[f"train_{metric}"][epoch_slice], label="Train")
    ax.plot(epochs, history[f"val_{metric}"][epoch_slice], label="Validation")
    ax.set(title=title, xlabel="Epoch", ylabel=title)
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# score: 4.04008
